# Step 9 — Robustness Analysis (Sensitivity to k)

We test whether the XAI consensus genes are stable across different choices of k
(number of genes retained from signature extraction). If the same genes emerge
regardless of k, the finding is robust and not an artifact of an arbitrary threshold.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
from collections import Counter

from asd_pipeline_utils import (
    RANDOM_STATE, ROBUSTNESS_PATH,
    get_targets, load_analysis_frame, load_signature_stability,
    split_features_and_metadata, run_robustness_for_k,
)

final_df = load_analysis_frame()
X, meta = split_features_and_metadata(final_df)
_, y_multi = get_targets(meta)
stability_df = load_signature_stability()

k_values = [50, 80, 100, 150]
consensus_per_k = {}

for k_val in k_values:
    print(f"\n{'='*50}")
    print(f"Running pipeline with k = {k_val}")
    print(f"{'='*50}")
    consensus, test_acc = run_robustness_for_k(X, y_multi, stability_df, k_val)
    consensus_per_k[k_val] = consensus
    print(f"  Test accuracy: {test_acc:.3f}")
    print(f"  Consensus genes (all 5 methods): {len(consensus)}")
    print(f"  -> {sorted(consensus)}")

# Summary
print(f"\n{'='*50}")
print("ROBUSTNESS SUMMARY")
print(f"{'='*50}\n")

for k_val in k_values:
    print(f"k={k_val:>3d}: {len(consensus_per_k[k_val]):>2d} consensus genes")

core_genes = set.intersection(*consensus_per_k.values())
print(f"\n*** Core genes (ALL k values): {len(core_genes)} ***")
print(f"    {sorted(core_genes)}")

gene_k_counts = Counter()
for genes in consensus_per_k.values():
    gene_k_counts.update(genes)

robust_genes = {g for g, c in gene_k_counts.items() if c >= 3}
print(f"\nGenes in >= 3/4 k values: {len(robust_genes)}")
print(f"    {sorted(robust_genes)}")

# Save robustness table
robust_table = pd.DataFrame([
    {"gene": g, "k_values_in_consensus": c,
     "in_k": ",".join(str(k) for k in k_values if g in consensus_per_k[k])}
    for g, c in gene_k_counts.most_common()
]).sort_values(["k_values_in_consensus", "gene"], ascending=[False, True]).reset_index(drop=True)

robust_table.to_csv(ROBUSTNESS_PATH, index=False)
print(f"\nSaved to {ROBUSTNESS_PATH}")
print(f"\n{robust_table.to_string(index=False)}")

if len(core_genes) >= 5:
    print(f"\nVERDICT: Choice of k does NOT significantly affect core findings.")
    print(f"         {len(core_genes)} genes robust across all k values.")
else:
    print(f"\nVERDICT: Some sensitivity to k. {len(core_genes)} genes stable across all k.")
    print(f"         {len(robust_genes)} genes stable in >= 3/4 k values.")